# Sales-led budget: what attribution and incrementality say together

Analysis behind [`docs/memo.md`](../docs/memo.md). Queries the dbt marts in BigQuery (`fct_attribution`, `rpt_attribution_by_channel`, `int_incrementality_factors`).

**To use in Hex:** import this `.ipynb`, attach the BigQuery connection, and replace `run(...)` with SQL cells (the SQL is unchanged).

In [ ]:
# Setup. In Hex, connect the BigQuery data connection instead and swap run() for SQL cells.
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJECT, DATASET = "mktg-portfolio", "dbt_dev"
AS_OF = "2026-09-30"          # end of the data period; tests past their re-test date are stale
SAVE_DIR = "../docs/memo"     # set to None to skip saving PNGs

client = bigquery.Client(project=PROJECT)

def table(name):
    return f"`{PROJECT}.{DATASET}.{name}`"

def run(sql):
    return client.query(sql).to_dataframe()

## Data

In [ ]:
# Attribution by channel and model (from the dbt report model).
channel_models = run(f"""
select attribution_model, channel_name, channel_group, spend_usd, leads, opportunities,
       pipeline_amount, share_of_pipeline, pipeline_per_spend_dollar
from {table('rpt_attribution_by_channel')}
""")

# Sales-led campaigns: spend vs. pipeline under last touch and position-based (W-shaped).
slg_campaigns = run(f"""
with spend as (
    select c.campaign_key, c.campaign_name, c.platform, m.product_line, sum(s.spend_usd) as spend
    from {table('fct_ad_spend')} s
    join {table('dim_campaign')} c on s.campaign_key = c.campaign_key
    join {table('campaign_motion_mapping')} m
        on c.platform = m.platform and c.campaign_id = cast(m.campaign_id as string)
    where m.motion = 'slg'
    group by 1, 2, 3, 4
),
credited as (
    select campaign_key,
           sum(credit_last_touch     * coalesce(pipeline_amount, 0)) as pipeline_last_touch,
           sum(credit_position_based * coalesce(pipeline_amount, 0)) as pipeline_position_based,
           sum(credit_position_based * case when is_opportunity then 1 else 0 end) as opps_position_based
    from {table('fct_attribution')}
    group by 1
)
select s.campaign_name, s.platform, s.product_line, s.spend,
       c.pipeline_last_touch, c.pipeline_position_based, c.opps_position_based
from spend s
left join credited c on s.campaign_key = c.campaign_key
""")

# Latest incrementality factor per channel, flagged stale past its re-test date.
factors = run(f"""
select channel_name, test_id, incrementality_factor as factor,
       incrementality_factor_low as factor_low, incrementality_factor_high as factor_high,
       retest_by, retest_by < date '{AS_OF}' as is_stale
from {table('int_incrementality_factors')}
where is_latest_test
""")

slg_campaigns["ppd_last_touch"] = slg_campaigns.pipeline_last_touch / slg_campaigns.spend
slg_campaigns["ppd_position_based"] = slg_campaigns.pipeline_position_based / slg_campaigns.spend
slg_campaigns["channel_name"] = slg_campaigns.platform.map(
    {"google": "Paid Search", "meta": "Paid Social - Meta", "linkedin": "Paid Social - LinkedIn"})
slg_campaigns = slg_campaigns.merge(factors, on="channel_name", how="left")
for col in ["factor", "factor_low", "factor_high"]:
    slg_campaigns[col] = slg_campaigns[col].astype(float)
slg_campaigns["ppd_incremental"]      = slg_campaigns.ppd_position_based * slg_campaigns.factor
slg_campaigns["ppd_incremental_low"]  = slg_campaigns.ppd_position_based * slg_campaigns.factor_low
slg_campaigns["ppd_incremental_high"] = slg_campaigns.ppd_position_based * slg_campaigns.factor_high
slg_campaigns = slg_campaigns.sort_values("ppd_position_based").reset_index(drop=True)
slg_campaigns[["campaign_name", "spend", "ppd_last_touch", "ppd_position_based",
               "factor", "ppd_incremental", "ppd_incremental_low", "ppd_incremental_high", "is_stale"]].round(2)

In [ ]:
# Chart style: light surface, recessive axes, two categorical slots validated for color vision deficiency.
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, RANGE = "#2a78d6", "#eb6834", "#b9b8b2"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 10, "text.color": INK,
    "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.grid": True, "axes.grid.axis": "x",
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
})

def title(fig, text, sub=None):
    fig.suptitle(text, x=0.01, ha="left", fontsize=11.5, color=INK, y=0.98)
    if sub:
        fig.text(0.01, 0.915, sub, ha="left", fontsize=9, color=INK_2)

def finish(fig, name, top=0.90):
    fig.tight_layout(rect=(0, 0, 1, top))
    if SAVE_DIR:
        import os
        os.makedirs(SAVE_DIR, exist_ok=True)
        fig.savefig(f"{SAVE_DIR}/{name}.png", dpi=200)
    return fig

## 1. Last touch flatters the campaigns that capture demand

Brand search and retargeting reach people who are already looking for us. Last touch gives them all the credit for the form fill; position-based shares it with the touches that started the journey.

In [ ]:
# 1. Sales-led campaigns: pipeline per spend dollar, last touch vs. position-based.
d = slg_campaigns
y = range(len(d))
h = 0.36
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh([i + h/2 + 0.01 for i in y], d.ppd_last_touch, height=h, color=BLUE, label="Last touch")
ax.barh([i - h/2 - 0.01 for i in y], d.ppd_position_based, height=h, color=ORANGE, label="Position-based (W-shaped)")
for i, r in d.iterrows():
    ax.text(r.ppd_last_touch + 0.6, i + h/2, f"${r.ppd_last_touch:.0f}", va="center", fontsize=8.5, color=INK_2)
    ax.text(r.ppd_position_based + 0.6, i - h/2, f"${r.ppd_position_based:.0f}", va="center", fontsize=8.5, color=INK)
ax.set_yticks(list(y), [f"{n}  (${s/1000:.0f}K)" for n, s in zip(d.campaign_name, d.spend)])
ax.set_xlabel("Pipeline per $1 of spend")
title(fig, "Last touch flatters search and Meta retargeting",
      "Sales-led campaigns, pipeline per $1 of spend, Jul to Sep 2026 (quarterly spend in brackets)")
ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.tick_params(axis="y", length=0)
finish(fig, "01_slg_pipeline_per_dollar")

## 2. Pipeline share moves to the channels that start journeys

In [ ]:
# 2. Share of pipeline by channel: last touch vs. position-based.
p = (channel_models[channel_models.attribution_model.isin(["last_touch", "position_based"])]
     .pivot_table(index="channel_name", columns="attribution_model", values="share_of_pipeline"))
p = p[(p.max(axis=1) > 0.01)].sort_values("position_based")
fig, ax = plt.subplots(figsize=(8, 3.8))
for i, (ch, r) in enumerate(p.iterrows()):
    ax.plot([r.last_touch, r.position_based], [i, i], color=RANGE, linewidth=2, zorder=1)
    ax.scatter(r.last_touch, i, s=64, color=BLUE, zorder=2, edgecolors=SURFACE, linewidths=2,
               label="Last touch" if i == 0 else None)
    ax.scatter(r.position_based, i, s=64, color=ORANGE, zorder=3, edgecolors=SURFACE, linewidths=2,
               label="Position-based (W-shaped)" if i == 0 else None)
    lo, hi = sorted([r.last_touch, r.position_based])
    ax.text(hi + 0.008, i, f"{r.last_touch:.0%} → {r.position_based:.0%}", va="center", fontsize=8.5, color=INK_2)
ax.set_yticks(range(len(p)), p.index)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlim(0, max(p.max()) + 0.08)
ax.set_xlabel("Share of total pipeline")
title(fig, "Pipeline credit moves to the channels that start and nurture journeys",
      "Share of total pipeline, last touch vs. position-based (W-shaped)")
ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.tick_params(axis="y", length=0)
finish(fig, "02_pipeline_share_by_model")

## 3. Adjusting for incrementality

Attribution says where credit lands, not what the spend caused. Multiplying position-based pipeline per dollar by each channel's latest lift-test factor gives a directional incremental return. The factors are channel-level and measured on signups, so treat this as a ranking, not a forecast. Retargeting is usually less incremental than its channel average, so its estimates here are the most likely to be too high.

In [ ]:
# 3. Incrementality-adjusted pipeline per dollar: position-based x the channel's lift-test factor,
#    with the test's confidence range. Directional: factors are channel-level and signup-based.
d = slg_campaigns
fig, ax = plt.subplots(figsize=(8, 4.2))
for i, r in d.iterrows():
    ax.plot([r.ppd_incremental_low, r.ppd_incremental_high], [i, i], color=RANGE, linewidth=3,
            solid_capstyle="round", zorder=1)
    ax.scatter(r.ppd_incremental, i, s=70, color=BLUE, zorder=2, edgecolors=SURFACE, linewidths=2)
    ax.text(r.ppd_incremental_high + 1.0, i, f"${r.ppd_incremental:.0f}  (${r.ppd_incremental_low:.0f}–${r.ppd_incremental_high:.0f})",
            va="center", fontsize=8.5, color=INK_2)
ax.set_yticks(range(len(d)), d.campaign_name)
ax.set_xlim(0, d.ppd_incremental_high.max() * 1.25)
ax.set_xlabel("Incremental pipeline per $1 of spend (estimate and test range)")
stale = ", ".join(d.loc[d.is_stale, "channel_name"].unique())
title(fig, "After incrementality, LinkedIn prospecting still beats Meta retargeting",
      f"Position-based pipeline per $1 x lift-test factor; bars are the test's range. Past re-test date: {stale}")
ax.tick_params(axis="y", length=0)
finish(fig, "03_incremental_pipeline_per_dollar")

## 4. The proposed move

In [ ]:
# Scenario: move half of Meta Retargeting's quarterly spend to LinkedIn Thought Leadership Doc Ads.
# Uses average returns, which overstate the marginal return of extra spend: read it as an upper bound.
src = slg_campaigns.set_index("campaign_name").loc["Retargeting - Site Visitors 30d"]
dst = slg_campaigns.set_index("campaign_name").loc["Thought Leadership - Doc Ads"]
move = src.spend / 2
scenario = pd.DataFrame({
    "view": ["Position-based", "Incremental (estimate)", "Incremental (pessimistic)"],
    "pipeline_gained": [move * dst.ppd_position_based, move * dst.ppd_incremental, move * dst.ppd_incremental_low],
    "pipeline_lost":   [move * src.ppd_position_based, move * src.ppd_incremental, move * src.ppd_incremental_high],
})
scenario["net_change"] = scenario.pipeline_gained - scenario.pipeline_lost
print(f"Budget moved: ${move:,.0f} per quarter")
scenario.round(0)